# 🧠 ColonoMind — LIMUC Train & Deploy **V2** (clean, Colab-ready)

Self-contained, memory-light pipeline. Run top-to-bottom.

**What it does**
1. Mounts Google Drive; **everything** (dataset, weights, plots, predictions, app) is saved under `MyDrive/ColonoMind/`.
2. Downloads LIMUC **once to Drive** — skips the download if it's already there.
3. Trains all 5 hybrid models (ResNet-50, DenseNet-121, EfficientNet-B4, ConvNeXt-Tiny, ViT-B/16) for 1–2 epochs to save weights.
4. Saves loss/accuracy curves, classifies the test set, saves predictions (CSV + montage).
5. Generates + launches the Streamlit diagnostic app.

**Before you run**
- Runtime → Change runtime type → **GPU**.
- After **Cell 1** (installs, incl. `transformers==4.41.2`): **Runtime → Restart session**, then run from **Cell 2**.
- Small by design for limited Colab: `MAX_PER_CLASS_TRAIN=50`, `MAX_PER_CLASS_TEST=20`. Raise later for real training.

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# transformers==4.41.2 is the last version shipping TFViTModel; tf-keras is its
# Keras-2 backend. ⚠️ AFTER this cell: Runtime → Restart session, then run Cell 2.
!pip install -q streamlit joblib lightgbm umap-learn PyWavelets scikit-image \
               opencv-python-headless pillow tensorflow transformers==4.41.2 tf-keras \
               imbalanced-learn tqdm
# Optional public URL for the Streamlit app in Colab (Cell 13):
# !pip install -q pyngrok

In [ ]:
# ============================================================
# CELL 2 — Imports, config, Google Drive, paths  (run this first!)
# ============================================================
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # must be set BEFORE importing tensorflow

import json, zipfile, urllib.request, shutil, time
import numpy as np, pandas as pd, cv2, pywt, scipy.stats, joblib
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, cohen_kappa_score)
from imblearn.over_sampling import SMOTE
import umap
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, GlobalAveragePooling2D,
                                      BatchNormalization, Dropout, Concatenate, Lambda)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
# ALL model backbones imported here so re-running any later cell just works
from tensorflow.keras.applications import ResNet50, DenseNet121, EfficientNetB4, ConvNeXtTiny
try:
    from transformers import TFViTModel
    HAS_VIT = True
except Exception as _e:
    HAS_VIT = False
    print("ⓘ ViT-B/16 unavailable (transformers TF backend not loaded) — the 4 CNNs will still train.")
    print("  reason:", _e)

for _g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(_g, True)
    except Exception: pass
np.random.seed(42); tf.random.set_seed(42)

# ---------------- knobs ----------------
IMG_SIZE   = (224, 224)
WAVELET    = "db1"
EPOCHS     = 2
BATCH_SIZE = 8            # small = safe on limited GPU RAM
LR         = 1e-4
MAX_PER_CLASS_TRAIN = 50  # limited Colab; raise for real training (None = full)
MAX_PER_CLASS_TEST  = 20
IGNORE_KEYWORDS = ["augment", "mask", "seg", "._", "crop"]

# ---------------- Google Drive + paths ----------------
USE_DRIVE, DRIVE_SUBDIR = True, "ColonoMind"
def _in_colab():
    try: import google.colab; return True
    except Exception: return False
IN_COLAB = _in_colab()
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
    print("💾 Everything saves to Google Drive:", PROJECT_ROOT)
else:
    b = Path.cwd(); PROJECT_ROOT = b.parent if b.name == "streamlit_colonomind_agent" else b
    print("💾 Saving locally to:", PROJECT_ROOT)

DATA_ROOT   = PROJECT_ROOT                       # dataset persists on Drive
DATASET_DIR = DATA_ROOT / "Dataset" / "LIMUC"
DEPLOY_DIR  = PROJECT_ROOT / "streamlit_colonomind_agent"
WEIGHTS_DIR = DEPLOY_DIR / "weights"
PLOTS_DIR   = PROJECT_ROOT / "plots"
OUT_DIR     = PROJECT_ROOT / "outputs"
CACHE_DIR   = PROJECT_ROOT / "cache"
for d in (DATASET_DIR, DEPLOY_DIR, WEIGHTS_DIR, PLOTS_DIR, OUT_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"), "| ViT:", HAS_VIT)

In [ ]:
# ============================================================
# CELL 3 — Get LIMUC (download ONCE to Drive, then reuse)
# ============================================================
ZENODO = "https://zenodo.org/records/5827695/files/{name}?download=1"

def find_class_root(root):
    root = Path(root)
    if not root.exists(): return None
    for p in [root] + [d for d in root.rglob("*") if d.is_dir()]:
        subs = [c for c in p.iterdir() if c.is_dir()]
        if subs and any(any(f.suffix.lower() in (".bmp",".jpg",".jpeg",".png",".tif")
                            for f in s.iterdir() if f.is_file()) for s in subs):
            return p
    return None

def get_split(split):
    folder = DATASET_DIR / split
    root = find_class_root(folder)
    if root:
        print(f"  ✔ Found on Drive, skipping download — {split}")
        return root
    zpath = DATASET_DIR / f"{split}.zip"
    if not (zpath.exists() and zpath.stat().st_size > 0):
        print(f"  ↓ downloading {split}.zip to Drive (one time)...")
        t0 = time.time()
        def hook(b, bs, tot):
            if tot > 0 and b % 500 == 0:
                print(f"    {min(100,100*b*bs/tot):5.1f}%  ({b*bs/1e6:6.0f}/{tot/1e6:.0f} MB)", end="\r")
        urllib.request.urlretrieve(ZENODO.format(name=split+".zip"), zpath, reporthook=hook)
        print(f"\n  ✔ downloaded in {time.time()-t0:.0f}s")
    print(f"  ↲ extracting {split}.zip ...")
    with zipfile.ZipFile(zpath) as zf: zf.extractall(folder)
    try: zpath.unlink()
    except Exception: pass
    return find_class_root(folder)

TRAIN_DIR = get_split("train_and_validation_sets")
TEST_DIR  = get_split("test_set")
CLASS_NAMES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
print("\nTRAIN_DIR :", TRAIN_DIR, "\nTEST_DIR  :", TEST_DIR, "\nCLASS_NAMES:", CLASS_NAMES)
assert len(CLASS_NAMES) >= 2, "Could not detect class folders."

_MAYO = {0:"normal or inactive disease (intact vascular pattern, no friability).",
         1:"mild activity (erythema, decreased vascular pattern, mild friability).",
         2:"moderate activity (marked erythema, absent vascular pattern, friability, erosions).",
         3:"severe activity (spontaneous bleeding, ulceration)."}
def _describe(name):
    d = next((int(c) for c in name if c.isdigit()), None)
    return f"Mayo Endoscopic Subscore {d} — {_MAYO.get(d,'')}" if d is not None else name
CLASS_DESCRIPTIONS = {c: _describe(c) for c in CLASS_NAMES}

In [ ]:
# ============================================================
# CELL 4 — Load images + 20 handcrafted features (17 DWT + 3 GLCM)
#          Caches to Drive so a restart reloads instantly.
# ============================================================
def extract_wavelet_stats(im):
    g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY); LL,(LH,HL,HH)=pywt.dwt2(g,WAVELET)
    st=lambda s:[np.mean(s),np.std(s),np.var(s),scipy.stats.entropy(np.abs(s.flatten())+1e-6)]
    return st(LL)+st(LH)+st(HL)+st(HH)+[np.sum(np.square(HH))]
def extract_glcm_features(im):
    g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)
    m = graycomatrix(g,[1,3,5],[0,np.pi/4,np.pi/2,3*np.pi/4],256,symmetric=True,normed=True)
    return [np.mean(graycoprops(m,"contrast")),np.mean(graycoprops(m,"homogeneity")),np.mean(graycoprops(m,"dissimilarity"))]
def extract_combined_features(im):
    f = extract_wavelet_stats(im)+extract_glcm_features(im); assert len(f)==20; return f
def load_dataset(dataset_dir, n=None):
    X,F,Y,P=[],[],[],[]
    for c in CLASS_NAMES:
        cd=Path(dataset_dir)/c
        if not cd.exists(): continue
        fs=[f for f in sorted(cd.iterdir()) if f.is_file() and not any(k in f.name.lower() for k in IGNORE_KEYWORDS)]
        for fp in (fs[:n] if n else fs):
            im=cv2.imread(str(fp))
            if im is None: continue
            im=cv2.cvtColor(cv2.resize(im,IMG_SIZE),cv2.COLOR_BGR2RGB)
            X.append(im); F.append(extract_combined_features(im)); Y.append(c); P.append(str(fp))
    return np.array(X), np.array(F), np.array(Y), P

RAW = CACHE_DIR / f"loaded_{MAX_PER_CLASS_TRAIN}_{MAX_PER_CLASS_TEST}.npz"
if RAW.exists():
    print("✔ Loading cached arrays from Drive (no feature recompute)...")
    z=np.load(RAW, allow_pickle=True)
    Xi_tr,Xf_tr,y_tr_lbl = z["Xi_tr"],z["Xf_tr"],z["y_tr_lbl"]
    Xi_te,Xf_te,y_te_lbl = z["Xi_te"],z["Xf_te"],z["y_te_lbl"]
    te_paths=list(z["te_paths"])
else:
    print("Loading train..."); Xi_tr,Xf_tr,y_tr_lbl,_        = load_dataset(TRAIN_DIR, MAX_PER_CLASS_TRAIN)
    print("Loading test..." ); Xi_te,Xf_te,y_te_lbl,te_paths = load_dataset(TEST_DIR,  MAX_PER_CLASS_TEST)
    np.savez_compressed(RAW, Xi_tr=Xi_tr,Xf_tr=Xf_tr,y_tr_lbl=y_tr_lbl,
                        Xi_te=Xi_te,Xf_te=Xf_te,y_te_lbl=y_te_lbl,te_paths=np.array(te_paths))
    print("✔ cached to Drive:", RAW.name)
print("train:", Xi_tr.shape, "| test:", Xi_te.shape)

In [ ]:
# ============================================================
# CELL 5 — Preprocess (MEMORY-SAFE): scale, SMOTE, UMAP, tf.data
#          Images stay uint8; normalized to /255 per batch.
# ============================================================
le = LabelEncoder(); y_tr = le.fit_transform(y_tr_lbl); y_te = le.transform(y_te_lbl)
NUM_CLASSES = len(le.classes_)
scaler = StandardScaler(); Xf_tr_s = scaler.fit_transform(Xf_tr); Xf_te_s = scaler.transform(Xf_te)

Xf_tr_bal, y_tr_bal = SMOTE(random_state=42).fit_resample(Xf_tr_s, y_tr)
idx_bal = np.empty(len(Xf_tr_bal), dtype=np.int64)
for cls in np.unique(y_tr):
    ci = np.where(y_tr==cls)[0]; fc = Xf_tr_s[ci]
    for r in np.where(y_tr_bal==cls)[0]:
        idx_bal[r] = ci[np.argmin(np.linalg.norm(fc - Xf_tr_bal[r], axis=1))]

umap_reducer = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, random_state=42)
Xtr_umap = umap_reducer.fit_transform(Xf_tr_bal).astype(np.float32)
Xte_umap = umap_reducer.transform(Xf_te_s).astype(np.float32)
y_tr_cat = to_categorical(y_tr_bal, NUM_CLASSES).astype(np.float32)
y_te_cat = to_categorical(y_te,     NUM_CLASSES).astype(np.float32)

Xi_tr_t = tf.convert_to_tensor(Xi_tr)   # one uint8 copy of the original images
def _train_map(i, feat, um, y):
    return (tf.cast(tf.gather(Xi_tr_t, i), tf.float32)/255.0, feat, um), y
train_ds = (tf.data.Dataset.from_tensor_slices((idx_bal, Xf_tr_bal.astype(np.float32), Xtr_umap, y_tr_cat))
            .shuffle(min(4096, len(idx_bal))).map(_train_map, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
def _val_map(img, feat, um, y):
    return (tf.cast(img, tf.float32)/255.0, feat, um), y
val_ds = (tf.data.Dataset.from_tensor_slices((Xi_te, Xf_te_s.astype(np.float32), Xte_umap, y_te_cat))
          .map(_val_map, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

plt.figure(figsize=(6,5))
plt.scatter(Xtr_umap[:,0], Xtr_umap[:,1], c=y_tr_bal, cmap="viridis", s=8, alpha=.7)
plt.title("UMAP of balanced features"); plt.colorbar(label="class")
plt.savefig(PLOTS_DIR/"umap_projection.png", dpi=120, bbox_inches="tight"); plt.show()
print(f"✅ preprocess done — balanced train={len(idx_bal)}, classes={list(le.classes_)}")

In [ ]:
# ============================================================
# CELL 6 — Architecture + the 5 model branches (uses Cell-2 imports)
# ============================================================
def focal_loss(gamma=2.5, alpha=0.25):
    def loss(yt, yp):
        yp = tf.clip_by_value(yp, 1e-8, 1.0)
        ce = -yt*tf.math.log(yp); w = alpha*tf.math.pow(1-yp, gamma)
        return tf.reduce_mean(tf.reduce_sum(w*ce, axis=1))
    return loss
def _head(o, dr):
    x = GlobalAveragePooling2D()(o)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x); return Dropout(dr)(x)
def branch_resnet50(s, dr=0.5):
    i=Input(shape=s, name="image_input_cnn"); return Model(i,_head(ResNet50(weights="imagenet",include_top=False,input_tensor=i).output,dr),name="ResNet50_Branch")
def branch_densenet121(s, dr=0.5):
    i=Input(shape=s, name="image_input_cnn"); return Model(i,_head(DenseNet121(weights="imagenet",include_top=False,input_tensor=i).output,dr),name="DenseNet121_Branch")
def branch_efficientnetb4(s, dr=0.5):
    i=Input(shape=s, name="image_input_cnn"); return Model(i,_head(EfficientNetB4(weights="imagenet",include_top=False,input_tensor=i).output,dr),name="EfficientNetB4_Branch")
def branch_convnexttiny(s, dr=0.5):
    i=Input(shape=s, name="image_input_cnn"); return Model(i,_head(ConvNeXtTiny(weights="imagenet",include_top=False,input_tensor=i).output,dr),name="ConvNeXtTiny_Branch")
def branch_vitb16(s, dr=0.5):
    if not HAS_VIT:
        raise ImportError("ViT unavailable — install transformers==4.41.2 + tf-keras (Cell 1) and restart.")
    i=Input(shape=s, name="image_input_vit")
    vit=TFViTModel.from_pretrained("google/vit-base-patch16-224-in21k"); vit.trainable=False
    cls=Lambda(lambda t: t[:,0,:])(vit(pixel_values=i).last_hidden_state)
    x=Dense(512,activation="relu",kernel_regularizer=l2(0.01))(cls); x=BatchNormalization()(x)
    return Model(i, Dropout(dr)(x), name="ViT_B16_Branch")

def build_hybrid_model(branch_fn, num_classes, dr=0.5):
    ii=Input((224,224,3), name="image_input"); xc=branch_fn((224,224,3),dr)(ii)
    xc=Dense(64,activation="relu",kernel_regularizer=l2(0.01))(xc); xc=BatchNormalization()(xc); xc=Dropout(dr)(xc)
    fi=Input((20,), name="feat_input"); xf=Dense(64,activation="relu",kernel_regularizer=l2(0.01))(fi); xf=BatchNormalization()(xf); xf=Dropout(dr)(xf)
    ui=Input((2,), name="umap_input"); xu=Dense(32,activation="relu",kernel_regularizer=l2(0.01))(ui); xu=BatchNormalization()(xu); xu=Dropout(dr)(xu)
    x=Dense(128,activation="relu",kernel_regularizer=l2(0.01))(Concatenate()([xc,xf,xu])); x=Dropout(dr)(x)
    return Model([ii,fi,ui], Dense(num_classes,activation="softmax",name="hybrid_output")(x))

MODELS = {
    "ResNet-50":       {"key":"resnet50",       "branch":branch_resnet50},
    "DenseNet-121":    {"key":"densenet121",    "branch":branch_densenet121},
    "EfficientNet-B4": {"key":"efficientnetb4", "branch":branch_efficientnetb4},
    "ConvNeXt-Tiny":   {"key":"convnexttiny",   "branch":branch_convnexttiny},
    "ViT-B-16":        {"key":"vitb16",         "branch":branch_vitb16},
}
print("Models:", list(MODELS))

In [ ]:
# ============================================================
# CELL 7 — Train all 5 models (1–2 epochs) & save weights to Drive
# ============================================================
all_results, histories, trained = {}, {}, {}
for name, cfg in MODELS.items():
    print(f"\n{'='*56}\n🚀 {name}\n{'='*56}")
    try:
        tf.keras.backend.clear_session()
        model = build_hybrid_model(cfg["branch"], NUM_CLASSES)
        model.compile(optimizer=Adam(LR), loss=focal_loss(2.5,0.25), metrics=["accuracy"])
        hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2)
        model.save_weights(str(WEIGHTS_DIR / f"{cfg['key']}.weights.h5"))
        proba = model.predict(val_ds, verbose=0); pred = np.argmax(proba, axis=1)
        all_results[name] = {"y_true": y_te, "y_pred": pred, "y_proba": proba}
        histories[name] = hist.history; trained[name] = cfg["key"]
        print(f"✔ {name}: saved, test-acc={accuracy_score(y_te, pred):.3f}")
        del model
    except Exception as e:
        print(f"SKIPPED {name}: {type(e).__name__}: {e}")
print("\nTrained OK:", list(trained))

In [ ]:
# ============================================================
# CELL 8 — Save loss & accuracy (train vs val) curves to Drive
# ============================================================
for name, h in histories.items():
    key = MODELS[name]["key"]; ep = range(1, len(h["loss"])+1)
    fig, ax = plt.subplots(1, 2, figsize=(11,4))
    ax[0].plot(ep,h["loss"],"o-",label="train"); ax[0].plot(ep,h["val_loss"],"s-",label="val")
    ax[0].set_title(f"{name} — Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep,h["accuracy"],"o-",label="train"); ax[1].plot(ep,h["val_accuracy"],"s-",label="val")
    ax[1].set_title(f"{name} — Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(PLOTS_DIR/f"{key}_history.png", dpi=120, bbox_inches="tight"); plt.show()
if histories:
    plt.figure(figsize=(7,4))
    for name,h in histories.items(): plt.plot(range(1,len(h["val_accuracy"])+1), h["val_accuracy"], "o-", label=name)
    plt.title("Validation accuracy — all models"); plt.xlabel("epoch"); plt.ylabel("val acc"); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(PLOTS_DIR/"val_accuracy_comparison.png", dpi=120, bbox_inches="tight"); plt.show()
print("Saved plots to:", PLOTS_DIR)

In [ ]:
# ============================================================
# CELL 9 — Metrics + saved predictions (CSV) + labelled montage
# ============================================================
metrics_summary = {}
for name, res in all_results.items():
    yt,yp,pr = np.asarray(res["y_true"]),np.asarray(res["y_pred"]),np.asarray(res["y_proba"])
    cm = confusion_matrix(yt,yp,labels=list(range(NUM_CLASSES)))
    specs=[]
    for i in range(NUM_CLASSES):
        tn=cm.sum()-cm[i,:].sum()-cm[:,i].sum()+cm[i,i]; fp=cm[:,i].sum()-cm[i,i]; specs.append(tn/(tn+fp+1e-8))
    metrics_summary[name]={"accuracy":float(accuracy_score(yt,yp)),
        "precision_macro":float(precision_score(yt,yp,average="macro",zero_division=0)),
        "recall_macro":float(recall_score(yt,yp,average="macro",zero_division=0)),
        "specificity_macro":float(np.mean(specs)),
        "f1_macro":float(f1_score(yt,yp,average="macro",zero_division=0)),
        "qwk":float(cohen_kappa_score(yt,yp,weights="quadratic")) if len(set(yt))>1 else 0.0,
        "confusion_matrix":cm.tolist(),
        "classification_report":classification_report(yt,yp,target_names=CLASS_NAMES,zero_division=0,output_dict=True)}
    df=pd.DataFrame({"image":[Path(p).name for p in te_paths],
                     "true_label":[CLASS_NAMES[i] for i in yt],
                     "pred_label":[CLASS_NAMES[i] for i in yp],
                     "confidence":pr.max(axis=1)})
    for i,c in enumerate(CLASS_NAMES): df[f"p_{c}"]=pr[:,i]
    df.to_csv(OUT_DIR/f"predictions_{MODELS[name]['key']}.csv", index=False)
if metrics_summary:
    print(pd.DataFrame(metrics_summary).T[["accuracy","precision_macro","recall_macro","f1_macro","qwk"]])
    best=max(all_results, key=lambda n: metrics_summary[n]["accuracy"])
    yp=all_results[best]["y_pred"]; pr=all_results[best]["y_proba"]
    n=min(12,len(te_paths)); cols=4; rows=int(np.ceil(n/cols)); plt.figure(figsize=(3*cols,3*rows))
    for i in range(n):
        img=cv2.cvtColor(cv2.imread(te_paths[i]),cv2.COLOR_BGR2RGB); ok=(yp[i]==y_te[i])
        plt.subplot(rows,cols,i+1); plt.imshow(img); plt.axis("off")
        plt.title(f"T:{CLASS_NAMES[y_te[i]]}\nP:{CLASS_NAMES[yp[i]]} ({pr[i].max():.2f})", color=("green" if ok else "red"), fontsize=9)
    plt.suptitle(f"Sample predictions — {best}"); plt.tight_layout()
    plt.savefig(OUT_DIR/"sample_predictions.png", dpi=120, bbox_inches="tight"); plt.show()
    print("Saved predictions + montage to:", OUT_DIR)
else:
    print("No models trained — nothing to report.")

In [ ]:
# ============================================================
# CELL 10 — Export artifacts + deployment report to Drive
# ============================================================
joblib.dump({"scaler":scaler,"umap_reducer":umap_reducer,"label_encoder":le,
             "class_names":CLASS_NAMES,"img_size":(224,224),"wavelet":WAVELET},
            DEPLOY_DIR/"preprocess_artifacts.joblib")
model_registry = {n:{"key":trained[n],"weights_path":f"{trained[n]}.weights.h5"} for n in trained}
report = {"project_name":"ColonoMind Diagnostic Agent (LIMUC)","class_names":CLASS_NAMES,
          "class_descriptions":CLASS_DESCRIPTIONS,"model_registry":model_registry,"metrics_summary":metrics_summary}
json.dump(report, open(DEPLOY_DIR/"deployment_report.json","w"), indent=2)
print("✔ exported to", DEPLOY_DIR, "| models in app:", list(model_registry))

In [ ]:
# ============================================================
# CELL 11 — requirements.txt + README for the deploy folder
# ============================================================
(DEPLOY_DIR/"requirements.txt").write_text("""streamlit==1.37.1
tensorflow-cpu==2.15.1
transformers==4.41.2
tf-keras
numpy==1.26.4
scikit-learn==1.4.2
umap-learn==0.5.6
pandas==2.2.2
scipy==1.13.1
joblib==1.4.2
opencv-python-headless==4.10.0.84
scikit-image==0.23.2
PyWavelets==1.6.0
Pillow==10.4.0
# Optional Claude Q&A:  anthropic==0.34.2
""", encoding="utf-8")
(DEPLOY_DIR/"README.md").write_text("""# ColonoMind Diagnostic Agent (LIMUC)

Streamlit app classifying colonoscopy images into Mayo 0-3 with 5 hybrid models.

## Run locally
    cd streamlit_colonomind_agent
    pip install -r requirements.txt
    streamlit run app.py

Weights (weights/*.h5) are large — use Git LFS or host externally for
Streamlit Community Cloud. Research/educational tool, not a clinical diagnosis.
""", encoding="utf-8")
print("✔ wrote requirements.txt + README.md")

In [ ]:
# ============================================================
# CELL 12 — Write the Streamlit app (app.py) to Drive
# ============================================================
APP_PATH = DEPLOY_DIR / 'app.py'
app_code = r'''
import os
import json
import joblib
import numpy as np
import pandas as pd
import cv2
import pywt
import scipy.stats
import streamlit as st

from pathlib import Path
from PIL import Image
from skimage.feature import graycomatrix, graycoprops

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, BatchNormalization,
    Dropout, Concatenate, Lambda
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import ResNet50, DenseNet121, EfficientNetB4

try:
    from tensorflow.keras.applications import ConvNeXtTiny
    HAS_CONVNEXT = True
except Exception:
    HAS_CONVNEXT = False

try:
    from transformers import TFViTModel
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False

# ============================================================
# 0. Page config + paths
# ============================================================
st.set_page_config(
    page_title="ColonoMind Diagnostic Agent",
    page_icon="\U0001F9E0",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.title("\U0001F9E0 ColonoMind Diagnostic Agent")
st.caption(
    "Upload a colonoscopy image, pick a trained model, run the diagnosis, "
    "review the model's report and ask questions. "
    "Research/educational tool — not a clinical diagnosis."
)

BASE_DIR = Path(__file__).resolve().parent
WEIGHTS_DIR = BASE_DIR / "weights"
REPORT_PATH = BASE_DIR / "deployment_report.json"
PREPROCESS_PATH = BASE_DIR / "preprocess_artifacts.joblib"

# Clinical meaning of the Mayo Endoscopic Subscore classes (grounding for Q&A)
MES_INFO = {
    "MES0": "Mayo Endoscopic Subscore 0 — normal or inactive disease "
            "(intact vascular pattern, no friability).",
    "MES1": "Mayo Endoscopic Subscore 1 — mild activity "
            "(erythema, decreased vascular pattern, mild friability).",
    "MES2": "Mayo Endoscopic Subscore 2 — moderate activity "
            "(marked erythema, absent vascular pattern, friability, erosions).",
    "MES3": "Mayo Endoscopic Subscore 3 — severe activity "
            "(spontaneous bleeding, ulceration).",
}

# ============================================================
# 1. Load deployment report + preprocessing artifacts
# ============================================================
@st.cache_data
def load_report():
    with open(REPORT_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

@st.cache_resource
def load_preprocess_artifacts():
    return joblib.load(PREPROCESS_PATH)

report = load_report()
artifacts = load_preprocess_artifacts()

CLASS_NAMES = artifacts["class_names"]
IMG_SIZE = tuple(artifacts["img_size"])
WAVELET = artifacts.get("wavelet", "db1")
scaler = artifacts["scaler"]
umap_reducer = artifacts["umap_reducer"]

MODEL_REGISTRY = report["model_registry"]
METRICS_SUMMARY = report["metrics_summary"]
MES_INFO = report.get("class_descriptions", MES_INFO)

# ============================================================
# 2. Feature extraction  (MUST match the training notebook exactly)
#    17 DWT features (16 stats + HH energy) + 3 GLCM = 20 features
# ============================================================
def extract_wavelet_stats(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    LL, (LH, HL, HH) = pywt.dwt2(gray, WAVELET)

    def stats(subband):
        subband_abs = np.abs(subband.flatten()) + 1e-6
        return [
            float(np.mean(subband)),
            float(np.std(subband)),
            float(np.var(subband)),
            float(scipy.stats.entropy(subband_abs)),
        ]

    dwt_features = stats(LL) + stats(LH) + stats(HL) + stats(HH)   # 16
    dwt_features.append(float(np.sum(np.square(HH))))              # + HH energy -> 17
    return dwt_features

def extract_glcm_features(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    distances = [1, 3, 5]
    angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
    glcm = graycomatrix(gray, distances=distances, angles=angles,
                        levels=256, symmetric=True, normed=True)
    return [
        float(np.mean(graycoprops(glcm, "contrast"))),
        float(np.mean(graycoprops(glcm, "homogeneity"))),
        float(np.mean(graycoprops(glcm, "dissimilarity"))),
    ]                                                             # 3

def extract_combined_features(image_rgb):
    feats = extract_wavelet_stats(image_rgb) + extract_glcm_features(image_rgb)
    assert len(feats) == 20, f"Expected 20 features, got {len(feats)}"
    return np.array(feats, dtype=np.float32)

def preprocess_uploaded_image(uploaded_file):
    pil_img = Image.open(uploaded_file).convert("RGB")
    image_rgb_original = np.array(pil_img)

    image_resized = cv2.resize(image_rgb_original, IMG_SIZE).astype(np.uint8)

    image_input = (image_resized.astype(np.float32) / 255.0)[np.newaxis, ...]

    handcrafted = extract_combined_features(image_resized).reshape(1, -1)
    handcrafted_scaled = scaler.transform(handcrafted)
    umap_feat = umap_reducer.transform(handcrafted_scaled)

    return {
        "pil_image": pil_img,
        "image_input": image_input,
        "handcrafted_scaled": handcrafted_scaled,
        "umap_feat": umap_feat,
    }

# ============================================================
# 3. Model architecture  (identical to the training notebook)
# ============================================================
def _cnn_head(base_model, dropout_rate):
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return x

def create_resnet50_branch(input_shape, dropout_rate=0.5):
    inp = Input(shape=input_shape, name="image_input_cnn")
    base = ResNet50(weights=None, include_top=False, input_tensor=inp)
    return Model(inp, _cnn_head(base, dropout_rate), name="ResNet50_Branch")

def create_densenet121_branch(input_shape, dropout_rate=0.5):
    inp = Input(shape=input_shape, name="image_input_cnn")
    base = DenseNet121(weights=None, include_top=False, input_tensor=inp)
    return Model(inp, _cnn_head(base, dropout_rate), name="DenseNet121_Branch")

def create_efficientnetb4_branch(input_shape, dropout_rate=0.5):
    inp = Input(shape=input_shape, name="image_input_cnn")
    base = EfficientNetB4(weights=None, include_top=False, input_tensor=inp)
    return Model(inp, _cnn_head(base, dropout_rate), name="EfficientNetB4_Branch")

def create_convnexttiny_branch(input_shape, dropout_rate=0.5):
    if not HAS_CONVNEXT:
        raise ImportError("ConvNeXtTiny is not available in this Keras version.")
    inp = Input(shape=input_shape, name="image_input_cnn")
    base = ConvNeXtTiny(weights=None, include_top=False, input_tensor=inp)
    return Model(inp, _cnn_head(base, dropout_rate), name="ConvNeXtTiny_Branch")

def create_vitb16_branch(input_shape, dropout_rate=0.5):
    if not HAS_TRANSFORMERS:
        raise ImportError("transformers is not installed — needed for ViT-B-16.")
    inp = Input(shape=input_shape, name="image_input_vit")
    vit = TFViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
    vit.trainable = False
    outputs = vit(pixel_values=inp)
    cls_token = Lambda(lambda t: t[:, 0, :])(outputs.last_hidden_state)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(cls_token)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inp, x, name="ViT_B16_Branch")

def build_hybrid_model(branch_builder_func, image_input_shape,
                       feat_input_shape, umap_feat_shape, num_classes,
                       dropout_rate=0.5):
    image_input = Input(shape=image_input_shape, name="image_input")
    x_cnn = branch_builder_func(image_input_shape, dropout_rate)(image_input)
    x_cnn = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x_cnn)
    x_cnn = BatchNormalization()(x_cnn)
    x_cnn = Dropout(dropout_rate)(x_cnn)

    feat_input = Input(shape=feat_input_shape, name="feat_input")
    x_feat = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(feat_input)
    x_feat = BatchNormalization()(x_feat)
    x_feat = Dropout(dropout_rate)(x_feat)

    umap_input = Input(shape=umap_feat_shape, name="umap_input")
    x_umap = Dense(32, activation="relu", kernel_regularizer=l2(0.01))(umap_input)
    x_umap = BatchNormalization()(x_umap)
    x_umap = Dropout(dropout_rate)(x_umap)

    combined = Concatenate()([x_cnn, x_feat, x_umap])
    x = Dense(128, activation="relu", kernel_regularizer=l2(0.01))(combined)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation="softmax", name="hybrid_output")(x)

    return Model(inputs=[image_input, feat_input, umap_input], outputs=output)

BRANCH_BUILDERS = {
    "ResNet-50": create_resnet50_branch,
    "DenseNet-121": create_densenet121_branch,
    "EfficientNet-B4": create_efficientnetb4_branch,
    "ConvNeXt-Tiny": create_convnexttiny_branch,
    "ViT-B-16": create_vitb16_branch,
}

@st.cache_resource(show_spinner=False)
def load_selected_model(model_name):
    model = build_hybrid_model(
        branch_builder_func=BRANCH_BUILDERS[model_name],
        image_input_shape=(224, 224, 3),
        feat_input_shape=(20,),
        umap_feat_shape=(2,),
        num_classes=len(CLASS_NAMES),
        dropout_rate=0.5,
    )
    weight_path = WEIGHTS_DIR / MODEL_REGISTRY[model_name]["weights_path"]
    if not weight_path.exists():
        raise FileNotFoundError(f"Weight file not found: {weight_path}")
    model.load_weights(str(weight_path))
    return model

# ============================================================
# 4. Sidebar  —  model (weight) dropdown
# ============================================================
st.sidebar.header("⚙️ Model selection")
selected_model_name = st.sidebar.selectbox(
    "Choose which trained weight to run",
    list(MODEL_REGISTRY.keys()),
)
st.sidebar.success(f"Active model: {selected_model_name}")

use_llm = st.sidebar.checkbox("Use Claude for Q&A (optional)", value=False)
api_key_input = ""
if use_llm:
    api_key_input = st.sidebar.text_input(
        "ANTHROPIC_API_KEY", type="password",
        help="Leave empty to use the ANTHROPIC_API_KEY environment variable.",
    )

# ============================================================
# PART 1  —  Upload image  (top)
# ============================================================
st.markdown("---")
st.header("1️⃣ Upload image")

uploaded_file = st.file_uploader(
    "Upload a colonoscopy image",
    type=["png", "jpg", "jpeg", "bmp", "tif", "tiff"],
)

processed = None
if uploaded_file is not None:
    processed = preprocess_uploaded_image(uploaded_file)
    st.image(processed["pil_image"], caption="Uploaded image", width=380)
else:
    st.info("Upload an image to begin.")

# ============================================================
# PART 2  —  Chosen-model report (metrics)  (middle)
# ============================================================
st.markdown("---")
st.header("2️⃣ Model performance report")

if selected_model_name in METRICS_SUMMARY:
    m = METRICS_SUMMARY[selected_model_name]
    c1, c2, c3, c4, c5, c6 = st.columns(6)
    c1.metric("Accuracy", f"{m['accuracy']:.3f}")
    c2.metric("Precision", f"{m['precision_macro']:.3f}")
    c3.metric("Recall/Sens.", f"{m['recall_macro']:.3f}")
    c4.metric("Specificity", f"{m['specificity_macro']:.3f}")
    c5.metric("F1-score", f"{m['f1_macro']:.3f}")
    c6.metric("QWK", f"{m['qwk']:.3f}")

    cm = np.array(m["confusion_matrix"])
    cm_df = pd.DataFrame(
        cm,
        index=[f"True {c}" for c in CLASS_NAMES],
        columns=[f"Pred {c}" for c in CLASS_NAMES],
    )
    st.markdown("**Confusion matrix (validation set)**")
    st.dataframe(cm_df, use_container_width=True)
    st.caption(
        "Metrics reflect this model's evaluation in the training notebook."
    )
else:
    st.warning("No metrics found for this model in deployment_report.json.")

# ============================================================
# PART 3  —  Image + classification result
# ============================================================
st.markdown("---")
st.header("3️⃣ Run diagnosis & result")

if processed is None:
    st.info("Please upload an image first.")
elif st.button("\U0001F50D Run diagnosis", type="primary"):
    with st.spinner(f"Running {selected_model_name}..."):
        model = load_selected_model(selected_model_name)
        y_proba = model.predict(
            [processed["image_input"],
             processed["handcrafted_scaled"],
             processed["umap_feat"]],
            verbose=0,
        )[0]

    pred_idx = int(np.argmax(y_proba))
    pred_class = CLASS_NAMES[pred_idx]
    confidence = float(y_proba[pred_idx])

    st.session_state["last_prediction"] = {
        "model": selected_model_name,
        "predicted_class": pred_class,
        "confidence": confidence,
        "probabilities": {CLASS_NAMES[i]: float(y_proba[i])
                          for i in range(len(CLASS_NAMES))},
    }

# Persist the result across chat interactions
if "last_prediction" in st.session_state:
    pred = st.session_state["last_prediction"]
    left, right = st.columns([1, 1])
    with left:
        if processed is not None:
            st.image(processed["pil_image"], caption="Diagnosed image", width=340)
    with right:
        st.subheader("Classification result")
        st.success(f"Predicted class: **{pred['predicted_class']}**")
        st.metric("Confidence", f"{pred['confidence']:.3f}")
        st.caption(MES_INFO.get(pred["predicted_class"], ""))
        prob_df = pd.DataFrame({
            "Class": list(pred["probabilities"].keys()),
            "Probability": list(pred["probabilities"].values()),
        })
        st.bar_chart(prob_df.set_index("Class"))
        st.caption(f"Prediction produced by: {pred['model']}")

# ============================================================
# PART 4  —  Question & Answer agent  (bottom)
# ============================================================
st.markdown("---")
st.header("4️⃣ Diagnostic Q&A agent")
st.caption(
    "Ask about the selected model's metrics, the MES classes, or the last "
    "prediction. Answers are grounded in the deployment report."
)

if "chat_history" not in st.session_state:
    st.session_state["chat_history"] = []

def build_context():
    return {
        "selected_model": selected_model_name,
        "class_names": CLASS_NAMES,
        "class_descriptions": MES_INFO,
        "selected_model_metrics": METRICS_SUMMARY.get(selected_model_name, {}),
        "last_prediction": st.session_state.get("last_prediction"),
    }

def deterministic_answer(query, ctx):
    q = query.lower()
    m = ctx["selected_model_metrics"]
    pred = ctx["last_prediction"]
    name = ctx["selected_model"]

    metric_keys = [
        ("accuracy", "accuracy", "accuracy"),
        ("precision", "precision_macro", "macro precision"),
        ("specificity", "specificity_macro", "macro specificity"),
        ("f1", "f1_macro", "macro F1-score"),
        ("qwk", "qwk", "quadratic weighted kappa"),
    ]
    for kw, key, label in metric_keys:
        if kw in q:
            val = m.get(key)
            return (f"**{name}** — {label}: "
                    f"{val:.4f}" if isinstance(val, (int, float))
                    else f"{label} is not available for {name}.")
    if "recall" in q or "sensitivity" in q:
        val = m.get("recall_macro")
        return f"**{name}** — macro recall/sensitivity: {val:.4f}" \
               if isinstance(val, (int, float)) else "Recall not available."
    if "confusion" in q:
        return f"Confusion matrix for **{name}**: {m.get('confusion_matrix')}"
    if any(k in q for k in ["mes", "class", "meaning", "severity", "score"]):
        return "\n\n".join(f"- **{k}**: {v}" for k, v in ctx["class_descriptions"].items())
    if any(k in q for k in ["prediction", "result", "diagnos", "predict"]):
        if pred is None:
            return "No prediction yet — upload an image and click **Run diagnosis**."
        desc = ctx["class_descriptions"].get(pred["predicted_class"], "")
        return (f"Latest prediction ({pred['model']}): **{pred['predicted_class']}** "
                f"at {pred['confidence']:.3f} confidence.\n\n{desc}\n\n"
                "This is model output, not a clinical diagnosis.")
    return ("I can answer about this model's accuracy, precision, recall, "
            "specificity, F1, QWK, confusion matrix, the MES classes, and the "
            "latest prediction.")

def claude_answer(query, ctx, api_key):
    try:
        from anthropic import Anthropic
        client = Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
        msg = client.messages.create(
            model="claude-sonnet-5",
            max_tokens=600,
            system=(
                "You are a diagnostic assistant for the ColonoMind app. "
                "Answer ONLY from the provided JSON context. Do not invent "
                "metrics or make clinical decisions. State that outputs are "
                "model classifications, not a final diagnosis. Be concise."
            ),
            messages=[{
                "role": "user",
                "content": f"Context JSON:\n{json.dumps(ctx, indent=2)}\n\nQuestion: {query}",
            }],
        )
        return msg.content[0].text
    except Exception as e:
        return deterministic_answer(query, ctx) + f"\n\n_(Claude unavailable: {e})_"

for msg in st.session_state["chat_history"]:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_question = st.chat_input("Ask about this model or the prediction...")
if user_question:
    st.session_state["chat_history"].append({"role": "user", "content": user_question})
    with st.chat_message("user"):
        st.markdown(user_question)
    ctx = build_context()
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            if use_llm:
                answer = claude_answer(user_question, ctx, api_key_input)
            else:
                answer = deterministic_answer(user_question, ctx)
        st.markdown(answer)
    st.session_state["chat_history"].append({"role": "assistant", "content": answer})
'''
APP_PATH.write_text(app_code, encoding='utf-8')
print('✔ app written to:', APP_PATH)

In [ ]:
# ============================================================
# CELL 13 — Launch the Streamlit app
# ============================================================
APP_PATH = DEPLOY_DIR / "app.py"
print("Local:  streamlit run", APP_PATH)

# --- Colab public URL (uncomment one) ---
# Option 1 — ngrok (free token: https://dashboard.ngrok.com)
#   !pip install -q pyngrok
#   from pyngrok import ngrok
#   ngrok.kill(); ngrok.set_auth_token("YOUR_TOKEN")
#   print("Open:", ngrok.connect(8501))
#   !streamlit run {APP_PATH} --server.port 8501 --server.headless true &>/content/st.log &
#
# Option 2 — localtunnel (no account; password = the printed IP)
#   !streamlit run {APP_PATH} --server.port 8501 --server.headless true &>/content/st.log &
#   !npx --yes localtunnel --port 8501 & curl -s https://loca.lt/mytunnelpassword